<a href="https://colab.research.google.com/github/Mikhailo88/MN-Praktychni-K4/blob/main/LR_9_Rec_Sist_OlkhovenkoFIT4_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.0 MB/s  0:00:00


1. Доцільна реалізація

In [3]:
from surprise import Dataset, Reader
from surprise import SVD, KNNBasic
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy
from surprise.model_selection import train_test_split

import sys
import io

In [4]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip

!unzip -o ml-100k.zip

--2026-11-11 11:45:25--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  1.09MB/s    in 4.9s    

2026-11-11 11:45:31 (977 KB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.bas

In [5]:
import pandas as pd

data = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
data.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [6]:
reader = Reader(rating_scale=(1, 5))
data_surprise = Dataset.load_from_df(data[['user_id', 'movie_id', 'rating']], reader)

In [7]:
trainset, testset = train_test_split(data_surprise, test_size=0.2)

# 3. GridSearchCV для SVD
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3)
grid_search_svd.fit(data_surprise)

print("Best RMSE for SVD:", grid_search_svd.best_score['rmse'])
print("Best params for SVD:", grid_search_svd.best_params['rmse'])

# 4. GridSearchCV для KNNBasic
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

# Приглушення виводу similarity matrix
sys_stdout_backup = sys.stdout
sys.stdout = io.StringIO()

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3)
grid_search_knn.fit(data_surprise)

# Повернення нормального виводу
sys.stdout = sys_stdout_backup

print("Best RMSE for KNNBasic:", grid_search_knn.best_score['rmse'])
print("Best params for KNNBasic:", grid_search_knn.best_params['rmse'])

# 5. Найкращий SVD
best_svd = SVD(**grid_search_svd.best_params['rmse'])
best_svd.fit(trainset)
predictions_svd = best_svd.test(testset)
print("Test RMSE (SVD):", accuracy.rmse(predictions_svd))

# 6. Найкращий KNNBasic
best_knn = KNNBasic(**grid_search_knn.best_params['rmse'])
best_knn.fit(trainset)
predictions_knn = best_knn.test(testset)
print("Test RMSE (KNNBasic):", accuracy.rmse(predictions_knn))

Best RMSE for SVD: 0.9316793942290168
Best params for SVD: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}
Best RMSE for KNNBasic: 1.0194712186570116
Best params for KNNBasic: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'pearson', 'user_based': True}}
RMSE: 0.9221
Test RMSE (SVD): 0.9220817664678822
Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 1.0150
Test RMSE (KNNBasic): 1.0150236425142443


In [8]:
best_svd = grid_search_svd.best_estimator['mae']
best_knn = grid_search_knn.best_estimator['mae']

print(f"Найкращий MAE для SVD: {grid_search_svd.best_score['mae']}")
print(f"Найкращий MAE для KNNBasic: {grid_search_knn.best_score['mae']}")

# Вибір моделі з найменшим MAE
if grid_search_svd.best_score['mae'] < grid_search_knn.best_score['mae']:
    best_model = best_svd
    print("Вибраний алгоритм: SVD")
else:
    best_model = best_knn
    print("Вибраний алгоритм: KNNBasic")



Найкращий MAE для SVD: 0.7346260291436456
Найкращий MAE для KNNBasic: 0.8093683337135914
Вибраний алгоритм: SVD


In [9]:
for i, (uid, iid, rating) in enumerate(trainset.all_ratings()):
    print(uid, iid, rating)
    if i > 10:
        break

0 0 3.0
0 129 4.0
0 176 4.0
0 251 3.0
0 624 4.0
0 240 4.0
0 193 4.0
0 177 4.0
0 396 5.0
0 67 2.0
0 120 3.0
0 643 4.0


In [13]:
# Навчання найкращої моделі
best_model.fit(trainset)

# Вибираємо реального користувача з trainset
inner_uid = next(iter(trainset.all_users()))
user_id = trainset.to_raw_uid(inner_uid)

# Оцінки цього користувача
user_ratings = trainset.ur[inner_uid]

print(f"Кількість рецензій користувача {user_id}: {len(user_ratings)}")

# Усі фільми та вже оцінені фільми
all_items = set(trainset.all_items())
rated_items = set([item for (item, _) in user_ratings])
unrated_items = all_items - rated_items

# Прогнозування рейтингів для неоцінених фільмів
predictions = [
    (item, best_model.predict(user_id, trainset.to_raw_iid(item)).est)
    for item in unrated_items
]

# Сортування за рейтингом
predictions.sort(key=lambda x: x[1], reverse=True)

# Виведення 10 найкращих фільмів
print("Топ-10 фільмів, рекомендованих для користувача:")

for item_id, rating in predictions[:10]:
    print(f"Фільм {trainset.to_raw_iid(item_id)} з прогнозованим рейтингом {rating:.2f}")

Кількість рецензій користувача 363: 249
Топ-10 фільмів, рекомендованих для користувача:
Фільм 223 з прогнозованим рейтингом 4.38
Фільм 64 з прогнозованим рейтингом 4.37
Фільм 483 з прогнозованим рейтингом 4.22
Фільм 357 з прогнозованим рейтингом 4.20
Фільм 645 з прогнозованим рейтингом 4.20
Фільм 1169 з прогнозованим рейтингом 4.19
Фільм 166 з прогнозованим рейтингом 4.14
Фільм 611 з прогнозованим рейтингом 4.13
Фільм 480 з прогнозованим рейтингом 4.13
Фільм 178 з прогнозованим рейтингом 4.09


In [16]:
print('214' in trainset._raw2inner_id_users)

False


In [22]:

best_model.fit(trainset)

# Виведення рекомендацій для конкретного користувача
inner_uid = next(iter(trainset.all_users()))
user_id = trainset.to_raw_uid(inner_uid)
user_ratings = trainset.ur[inner_uid]
print(f"Кількість рецензій користувача {user_id}: {len(user_ratings)}")

# Отримання рекомендацій
all_items = set(trainset.all_items())
rated_items = set([item for (item, _) in user_ratings])
unrated_items = all_items - rated_items

# Прогнозування рейтингів для нерецензованих фільмів
predictions = [(item, best_model.predict(user_id, trainset.to_raw_iid(item)).est) for item in unrated_items]
predictions.sort(key=lambda x: x[1], reverse=True)

# Виведення 10 найкращих фільмів
print("Топ-10 фільмів, рекомендованих для користувача:")
for item_id, rating in predictions[:10]:
    print(f"Фільм {trainset.to_raw_iid(item_id)} з прогнозованим рейтингом {rating:.2f}")

Кількість рецензій користувача 363: 249
Топ-10 фільмів, рекомендованих для користувача:
Фільм 268 з прогнозованим рейтингом 4.41
Фільм 483 з прогнозованим рейтингом 4.21
Фільм 246 з прогнозованим рейтингом 4.16
Фільм 357 з прогнозованим рейтингом 4.16
Фільм 192 з прогнозованим рейтингом 4.15
Фільм 519 з прогнозованим рейтингом 4.13
Фільм 160 з прогнозованим рейтингом 4.13
Фільм 318 з прогнозованим рейтингом 4.12
Фільм 641 з прогнозованим рейтингом 4.08
Фільм 64 з прогнозованим рейтингом 4.07
